# Ledger Eyes: what is the stock crowd saying, and what are they looking at?

Stock forums are loud. This notebook reads a batch of posts and answers two questions. Is the crowd **bullish or bearish**? And are they **counting or telling stories**? The second answer is a rung on a six-level ladder, from "assets and dividends" at the bottom (close to cash) up to "emotion only" at the top.

The chain is the one from class: load → clean → tokenize → split → vectorize → neural network → evaluate. Everything runs from top to bottom on Google Colab, with no keys and no uploads.

***

# 账本的眼：股票人群在说什么、在看什么？

股票论坛很吵。这个笔记本读一批帖子，回答两个问题：人群**看多还是看空**；他们是在**算账还是讲故事**。第二个答案是六级梯子上的一级，从最底下的「资产和分红」（离现金最近）到最顶上的「纯情绪」。

整条链是课上教的顺序：加载 → 清洗 → 分词 → 切分 → 向量化 → 神经网络 → 评估。在 Google Colab 上从头跑到底，不需要钥匙，也不需要上传文件。

## Settings

Everything that can be changed is in the first cell. The second cell installs the three packages Colab does not ship, then imports the libraries.

***

## 可调参数

能改的东西全在第一格，改完以后从头再运行一遍。两个地址是数据来源：38,091 条推文在 Hugging Face，8,000 条六级标签和 200 条复核在本项目公开的 GitHub 仓库。`HIDDEN_LAYERS` 是网络中间层的大小，写 `()` 就没有中间层。六级网络只训练 4 遍，因为它的数据少，看多了会死记训练集（过拟合）。第二格先装 Colab 没有预装的三个包，再把后面要用的库引进来。

In [ ]:
# Everything that can be changed. Run the notebook from the top again after a change.
DATA_URL = ("hf://datasets/TimKoornstra/financial-tweets-sentiment@bba602698fd6f18226f6404fd8fb637ef67fa1eb"
            "/data/train-00000-of-00001.parquet")                      # 38,091 stock tweets with a sentiment label
REPO_URL = "https://raw.githubusercontent.com/Zhen131/ZHENNs_Ledger_Eyes/main/"
LEVEL_LABELS_URL = REPO_URL + "labels/levels-two-models-plus-374.csv"  # level labels of 30,479 of those tweets
LEVEL_REVIEW_URL = REPO_URL + "labels/levels-spot-check-200.csv"       # a blind second opinion on 200 of them
RANDOM_SEED = 42          # same seed, same samples and same training every run
VOCAB_SIZE = 5000         # the vectorizers keep the 5000 most frequent words
HIDDEN_LAYERS = (64, 32)  # units of the hidden layers, as in class; () means no hidden layer
EPOCHS = 5                # passes over the training set, sentiment networks
LEVEL_EPOCHS = 4          # passes over the training set, level network (little data: more passes overfit)
BATCH_SIZE = 50           # posts per training step

In [ ]:
# Install what Colab does not ship, then import everything used below.
%pip install -q emoji contractions gensim

import re
import warnings

import contractions
import emoji
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import tensorflow as tf
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

## 1. Load the data

A public dataset on Hugging Face: 38,091 short English posts about stocks, each labelled bearish, neutral or bullish. The three cells download it, take a first look, and draw the same number of posts from every class.

***

## 1. 加载数据

数据是 Hugging Face 上一份公开的股票推文集，每条带看空、中性、看多三种标签之一。第一格下载数据，整理成编号、原文、标签三列；编号就是六级标签文件用来对上帖子的那个编号。第二格数每一类有多少条，再画出每条帖子有多少个词。第三格从每一类里抽同样多的帖子，这样模型不会偏向帖子多的那一类。

In [ ]:
# Download the posts and keep three columns: id, text and label.
LABEL_NAMES = {0: "bearish", 1: "neutral", 2: "bullish"}
raw = pd.read_parquet(DATA_URL)
df_all = pd.DataFrame({
    "post_id": [f"tim:{row}" for row in range(len(raw))],   # the level label file finds a post by this id
    "text": raw["tweet"].fillna("").astype(str),
    "label": raw["sentiment"].map({0: 1, 1: 2, 2: 0}),       # in the source 0 is neutral, 1 bullish, 2 bearish
})
print(f"{len(df_all):,} posts")
df_all.head()

In [ ]:
# Count the posts of every class and draw the number of words per post.
print(df_all["label"].map(LABEL_NAMES).value_counts())
df_all["text"].str.count(r"\S+").clip(upper=60).hist(bins=60, figsize=(8, 3))
plt.xlabel("words per post")
plt.ylabel("posts")
plt.show()

In [ ]:
# Draw the same number of posts from every class, so that no class outweighs the others.
per_class = df_all["label"].value_counts().min()
df = pd.concat([df_all[df_all["label"] == label].sample(n=per_class, random_state=RANDOM_SEED) for label in LABEL_NAMES])
df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
print(f"{per_class:,} posts per class, {len(df):,} in all")

## 2. Clean the text

Eight steps, in the order from class. Negation words are kept on purpose: "not bullish" must not become "bullish". Emoji become words, because in stock posts they carry the mood.

***

## 2. 清洗

清洗分八步，顺序和课上一样：小写、去多余空格、去 HTML 标签、表情换成词、去特殊字符、展开缩写、去停用词、词形还原。否定词故意留下，因为 not bullish 不能变成 bullish。表情换成英文词而不是直接丢，因为股票帖子里表情带着情绪。第一格定义这八步；第二格拿一句话演示每走一步文字变成什么样；第三格清洗全部帖子，要等十几秒。

In [ ]:
# The eight cleaning steps, each a small function.
for package in ["stopwords", "punkt_tab", "wordnet"]:
    nltk.download(package, quiet=True)
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

STOP_WORDS = set(stopwords.words("english"))
NOT_FORMS = {word for word in STOP_WORDS if word.endswith("n't")}
STOP_WORDS -= {"no", "not", "nor", "never", "ain"} | NOT_FORMS | {word[:-3] + "n" for word in NOT_FORMS}
LEMMATIZER = WordNetLemmatizer()
SKIN_TONE = re.compile(r"[_,]*(?:medium-light|medium-dark|light|medium|dark)_skin_tone")


def emoji_to_words(text):
    """Replace every emoji with its English name, for example a rocket becomes " rocket "."""
    def name(chars, data):
        return " " + SKIN_TONE.sub("", data["en"].strip(":")).replace("_", " ").lower() + " "
    return emoji.replace_emoji(text, replace=name)


CLEANING_STEPS = {
    "lower case": str.lower,
    "extra whitespace": lambda text: " ".join(text.split()),
    "HTML tags": lambda text: BeautifulSoup(text, "html.parser").get_text(),
    "emoji to words": emoji_to_words,
    "special characters": lambda text: re.sub(r"[^a-zA-Z0-9\s]", "", text),
    "contractions": contractions.fix,
    "stop words": lambda text: " ".join(word for word in word_tokenize(text) if word.lower() not in STOP_WORDS),
    "lemmatization": lambda text: " ".join(LEMMATIZER.lemmatize(word) for word in word_tokenize(text)),
}


def clean(text):
    """Run one post through the eight steps and return the cleaned text."""
    for step in CLEANING_STEPS.values():
        text = step(text)
    return text

In [ ]:
# One sentence, step by step.
example = "I don't think $TSLA will pump, not bullish at all 😭"
print(f"{'original':20s}{example}")
for name, step in CLEANING_STEPS.items():
    example = step(example)
    print(f"{name:20s}{example}")

In [ ]:
# Clean every post of the sample (this takes some seconds).
df["clean_text"] = df["text"].map(clean)
df[["text", "clean_text"]].head()

## 3. Tokenize

Two ways to cut a post: whole words (NLTK) or pieces of words (BPE, trained here on these posts). The vectorizers below use whole words.

***

## 3. 分词

切帖子有两种切法：按整词切（NLTK），按词的片段切（BPE，在这批帖子上现场训练）。这一格把两句话用两种切法各切一遍，可以看到 BPE 把不常见的词（例如 Nvidia）拆成了几个片段。后面的向量化用的是整词。

In [ ]:
# Train a BPE tokenizer on the posts, then cut two sentences both ways.
bpe = Tokenizer(BPE(unk_token="[UNK]"))
bpe.pre_tokenizer = Whitespace()
bpe.train_from_iterator(df["text"].tolist(), BpeTrainer(vocab_size=5000, special_tokens=["[UNK]"], show_progress=False))
for text in ["Never selling my Nvidia shares 🚀🚀", "No way it can't go higher, I won't sell"]:
    print(text)
    print("  NLTK:", " | ".join(word_tokenize(text)))
    print("  BPE :", " | ".join(bpe.encode(text).tokens))

## 4. Split

80% to learn from, 10% to choose a model with, 10% that no model sees until the final score. Every part keeps the same share of each class.

***

## 4. 切成训练、验证、测试三份

八成用来学，一成用来挑模型，最后一成留到打分时才给模型看。每一份里各类的比例相同。

In [ ]:
def split_80_10_10(posts, label_column):
    """Split posts into training, validation and test sets (80 / 10 / 10), stratified by label_column."""
    train, rest = train_test_split(posts, test_size=0.2, random_state=RANDOM_SEED, stratify=posts[label_column])
    validation, test = train_test_split(rest, test_size=0.5, random_state=RANDOM_SEED, stratify=rest[label_column])
    return {"train": train, "validation": validation, "test": test}


parts = split_80_10_10(df, "label")
y = {part: frame["label"].to_numpy() for part, frame in parts.items()}
print({part: len(frame) for part, frame in parts.items()})

## 5. Vectorize

A network only reads numbers, so every post becomes a row of numbers. Three ways from class: word counts, TF-IDF, and the average of word vectors (Word2Vec, trained here). All three learn from the training set only.

***

## 5. 向量化

网络只会算数字，所以每条帖子要变成一行数字。这里用课上教的三种办法：数每个词出现几次；TF-IDF（也是数词，但把到处都有的词的分量压低）；词向量取平均（Word2Vec，现场训练，每个词是 100 个数）。三种都只从训练集里学词表，验证集和测试集不参与。

In [ ]:
def average_word_vectors(texts):
    """One row per text: the mean vector of its words that have a vector; zeros when none has one."""
    vectors = np.zeros((len(texts), word2vec.vector_size), dtype=np.float32)
    for row, text in enumerate(texts):
        known = [word for word in text.split() if word in word2vec.wv.key_to_index]
        if known:
            vectors[row] = word2vec.wv[known].mean(axis=0)
    return vectors


# Learn the vocabularies and the word vectors from the training set, then turn all three parts into vectors.
count_vectorizer = CountVectorizer(max_features=VOCAB_SIZE, dtype=np.float32).fit(parts["train"]["clean_text"])
tfidf_vectorizer = TfidfVectorizer(max_features=VOCAB_SIZE, dtype=np.float32).fit(parts["train"]["clean_text"])
word2vec = Word2Vec([text.split() for text in parts["train"]["clean_text"]], vector_size=100, seed=RANDOM_SEED, workers=1)
VECTORIZERS = {
    "word counts": lambda texts: count_vectorizer.transform(texts).toarray(),
    "TF-IDF": lambda texts: tfidf_vectorizer.transform(texts).toarray(),
    "averaged word vectors": lambda texts: average_word_vectors(list(texts)),
}
X = {name: {part: to_vectors(frame["clean_text"]) for part, frame in parts.items()} for name, to_vectors in VECTORIZERS.items()}
for name in X:
    print(f"{name}: training matrix {X[name]['train'].shape}")

## 6. Neural network

The network from class: hidden layers with ReLU, then a softmax layer that gives every class a probability. One network is trained per kind of vectors.

***

## 6. 神经网络

网络和课上的一样：中间层用 ReLU，最后一层用 softmax 给每一类一个概率。中间层有几层、每层多大，看可调参数里的 `HIDDEN_LAYERS`。三种向量各训练一个网络，训练时每一遍都会打印训练集和验证集上的准确率。

In [ ]:
def train_network(X_train, y_train, X_val, y_val, n_classes, epochs):
    """Build a network (hidden layers as in HIDDEN_LAYERS, one softmax output per class), train it and return it."""
    tf.keras.utils.set_random_seed(RANDOM_SEED)
    layers = [tf.keras.Input(shape=(X_train.shape[1],))]
    layers += [tf.keras.layers.Dense(units, activation="relu") for units in HIDDEN_LAYERS]
    layers += [tf.keras.layers.Dense(n_classes, activation="softmax")]
    model = tf.keras.Sequential(layers)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(X_train, y_train, epochs=epochs, batch_size=BATCH_SIZE, validation_data=(X_val, y_val), verbose=2)
    return model


def predict(model, vectors):
    """The position of the most probable class, one per row."""
    return model.predict(vectors, batch_size=1024, verbose=0).argmax(axis=1)


models = {}
for name in X:
    print(name)
    models[name] = train_network(X[name]["train"], y["train"], X[name]["validation"], y["validation"], len(LABEL_NAMES), EPOCHS)

## 7. Evaluate

The score is macro-F1: one F1 per class, then the mean, so no class can be ignored. Two baselines show what guessing gives. The best network on the validation set is used from here on.

***

## 7. 评估

打分用 macro-F1：每一类单独算一个 F1 再取平均，所以哪一类都不能被忽略。两条对照线是「全猜同一类」和「乱猜」，网络的分数要比它们高才算学到了东西。第一格先用验证集挑出最好的网络，再在测试集上给三个网络和两条对照线打分。第二格画最好那个网络的混淆矩阵：行是真实标签，列是网络判的标签。

In [ ]:
def with_baselines(predictions, y_train, n_test):
    """Add two baselines to the predictions: always the most frequent class, and a uniform random guess."""
    nothing_train, nothing_test = np.zeros((len(y_train), 1)), np.zeros((n_test, 1))
    majority = DummyClassifier(strategy="most_frequent").fit(nothing_train, y_train)
    random_guess = DummyClassifier(strategy="uniform", random_state=RANDOM_SEED).fit(nothing_train, y_train)
    return {**predictions, "baseline: always one class": majority.predict(nothing_test),
            "baseline: random guess": random_guess.predict(nothing_test)}


# Choose the best network on the validation set, then score everything on the test set.
validation_f1 = {name: f1_score(y["validation"], predict(models[name], X[name]["validation"]), average="macro") for name in models}
best = max(validation_f1, key=validation_f1.get)
print("Validation macro-F1:", {name: round(score, 3) for name, score in validation_f1.items()}, "-> best:", best)

predictions = with_baselines({name: predict(models[name], X[name]["test"]) for name in models}, y["train"], len(y["test"]))
pd.DataFrame({name: {"accuracy": accuracy_score(y["test"], predicted), "macro-F1": f1_score(y["test"], predicted, average="macro")}
              for name, predicted in predictions.items()}).T.round(3)

In [ ]:
# Confusion matrix of the best network: rows are the true labels, columns what the network said.
ConfusionMatrixDisplay.from_predictions(y["test"], predictions[best], display_labels=list(LABEL_NAMES.values()), colorbar=False)
plt.title(f"Sentiment, test set ({best})")
plt.show()

## 8. The six levels: what does a post pay attention to?

The ladder climbs from numbers close to cash at the bottom up to pure emotion at the top: 1 assets & dividends, 2 earnings, 3 revenue & business, 4 price & chart, 5 story & macro, 6 emotion only; `none` means the post gives no reason about a stock.

No dataset has such labels. Two local language models (`gemma4:12b` and `gemma4:26b`) labelled all posts once, offline. The 30,105 posts on which they agree are kept, plus 374 posts of the smallest class that a third, larger model settled; the label file is in this project's public repository and is downloaded here. These are **silver labels**: given by a model, not by a person. The network below learns from them with the same chain as above.

***

## 8. 六级：帖子在关注哪一层？

梯子从最底下离现金近的数字，往上排到最顶上的纯情绪，共六级：1 资产和分红、2 利润、3 营收和业务、4 价格和图形、5 故事和宏观、6 纯情绪；`none` 表示帖子没有在讲股票的理由。

没有现成的数据集带这种标签，所以先让两个本地大模型（gemma4:12b 和 gemma4:26b）离线给全部帖子各打了一个标签。只留两个模型判得一样的 30,105 条，再加上最少那一类里由第三个更大的模型裁定的 374 条。标签文件放在本项目公开的仓库里，这里直接下载。这叫**银标签**：是模型打的，不是人打的。

第一格下载标签文件，按编号和帖子对上，再从七类里各抽同样多的帖子。第二格走一遍同样的链：清洗、切三份、数词、训练一个七个输出的网络。第三格在测试集上打分，平均分只算六级，`none` 不算进去。第四格画混淆矩阵，再报银标签的复核结果：第三个更大的模型在看不到原标签的情况下重标了其中 200 条，看它和原标签有多少条一致。

In [ ]:
# Download the label file, join it to the posts by id, and draw the same number of posts from each of the seven classes.
LEVELS = ["assets_dividends", "earnings", "revenue_business", "price_chart", "story_macro", "emotion_only"]
LEVEL_CLASSES = LEVELS + ["none"]

level_labels = pd.read_csv(LEVEL_LABELS_URL, keep_default_na=False).rename(columns={"label": "level"})
level_posts = level_labels.merge(df_all[["post_id", "text"]], on="post_id")
print(level_posts["level"].value_counts())

per_level = level_posts["level"].value_counts().min()
level_df = pd.concat([level_posts[level_posts["level"] == level].sample(n=per_level, random_state=RANDOM_SEED) for level in LEVEL_CLASSES])
level_df = level_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
print(f"{per_level} posts per class, {len(level_df):,} in all")

In [ ]:
# The same chain again: clean, split, count words, and train a network with seven outputs.
level_df["clean_text"] = level_df["text"].map(clean)
level_df["target"] = level_df["level"].map(LEVEL_CLASSES.index)    # positions 0 to 5 are the six levels, 6 is none
level_parts = split_80_10_10(level_df, "level")
level_vectorizer = CountVectorizer(max_features=VOCAB_SIZE, dtype=np.float32).fit(level_parts["train"]["clean_text"])
level_X = {part: level_vectorizer.transform(frame["clean_text"]).toarray() for part, frame in level_parts.items()}
level_y = {part: frame["target"].to_numpy() for part, frame in level_parts.items()}
level_model = train_network(level_X["train"], level_y["train"], level_X["validation"], level_y["validation"],
                            len(LEVEL_CLASSES), LEVEL_EPOCHS)

In [ ]:
# Score on the test set. The mean F1 is taken over the six levels; none is left out of it.
level_predictions = with_baselines({"word-count network": predict(level_model, level_X["test"])},
                                   level_y["train"], len(level_y["test"]))
six_levels = list(range(len(LEVELS)))
pd.DataFrame({name: {"accuracy": accuracy_score(level_y["test"], predicted),
                     "macro-F1 over the six levels": f1_score(level_y["test"], predicted, labels=six_levels, average="macro")}
              for name, predicted in level_predictions.items()}).T.round(3)

In [ ]:
# Confusion matrix of the level network, and how far a third model agrees with the silver labels.
ConfusionMatrixDisplay.from_predictions(level_y["test"], level_predictions["word-count network"], labels=list(range(len(LEVEL_CLASSES))),
                                        display_labels=["1", "2", "3", "4", "5", "6", "none"], colorbar=False)
plt.title("Levels, test set")
plt.show()

review = pd.read_csv(LEVEL_REVIEW_URL, keep_default_na=False)
same = review["review_label"] == review["label"]
on_ladder = review["label"] != "none"
print(f"How good are the silver labels? A third, larger language model re-labelled {len(review)} of the posts without seeing their labels.\n"
      f"It gave the same label for {same[on_ladder].mean():.0%} of the {on_ladder.sum()} posts on the six levels "
      f"and for {same[~on_ladder].mean():.0%} of the {(~on_ladder).sum()} posts labelled none.")

## 9. Use it: read a crowd

`read_the_crowd` gives every post two calls, one from each network. `show_the_crowd` counts them into two charts and a table: the share of each sentiment, the share of each level, and how the two meet. The cell reads the test posts of chapter 8 as one crowd.

***

## 9. 拿来用：读一群人

`read_the_crowd` 给每条帖子两个判断，两个网络各出一个。`show_the_crowd` 把它们数成两张图和一张表：多空各占几成，六级各占几成，两者怎么交叉。判成 `none` 的帖子不上梯子。这一格拿第 8 章测试集的帖子当一群人，读一遍。

In [ ]:
def read_the_crowd(posts):
    """Read raw posts with both networks: one row per post with its sentiment call and its level call."""
    cleaned = [clean(post) for post in posts]
    sentiment = predict(models[best], VECTORIZERS[best](cleaned))
    level = predict(level_model, level_vectorizer.transform(cleaned).toarray())
    return pd.DataFrame({"post": list(posts), "sentiment": [LABEL_NAMES[call] for call in sentiment],
                         "level": [LEVEL_CLASSES[call] for call in level]})


def show_the_crowd(calls):
    """Draw the sentiment shares and the level shares, and return the sentiment-by-level table (none left out)."""
    on_ladder = calls[calls["level"] != "none"]
    sentiment_names = list(LABEL_NAMES.values())
    figure, (left, right) = plt.subplots(1, 2, figsize=(12, 3.5))
    calls["sentiment"].value_counts(normalize=True).reindex(sentiment_names, fill_value=0).plot.bar(
        ax=left, rot=0, ylim=(0, 1), title=f"Sentiment: share of {len(calls)} posts")
    on_ladder["level"].value_counts(normalize=True).reindex(LEVELS, fill_value=0).plot.barh(
        ax=right, xlim=(0, 1), color="tab:orange",
        title=f"Ladder: share of {len(on_ladder)} posts ({len(calls) - len(on_ladder)} judged none)")
    plt.tight_layout()
    plt.show()
    return pd.crosstab(on_ladder["sentiment"], on_ladder["level"]).reindex(index=sentiment_names, columns=LEVELS, fill_value=0)


show_the_crowd(read_the_crowd(level_parts["test"]["text"]))

### Paste box

Put your own posts between the quotes, one per line, and run the two cells again.

***

### 粘贴框

把你自己的帖子贴到引号中间，一行一条，再运行下面两格。第一格列出每条帖子的两个判断，第二格画图和交叉表。

In [ ]:
MY_POSTS = """
$VZ yields almost seven percent and trades under book value, quietly collecting that dividend while everyone chases chips
$INTC margins got crushed again, earnings per share missed by a mile and the guidance looks thin
Cloud revenue at $MSFT grew faster than the rest of the business, new customers keep signing multiyear deals
$TSLA closed under the 200 day line on heavy volume, next support sits somewhere near 180
If the Fed holds rates through the winter, small caps stay stuck and $IWM goes nowhere
$NVDA to the moon, cannot stop smiling today, absolutely zero regrets
Everything is red again and my stomach hurts, I really hate this market
Rumor going around that a big tech name will announce a merger with a chip designer, story could get wild
Happy birthday to my little brother, dinner tonight at grandma's place
Free cash flow per share at $AAPL dwarfs the net income of most rivals, valuation still fair at this multiple
Watching the $COST report on Thursday, membership fee renewals matter more than store sales to me
"""
my_calls = read_the_crowd([line.strip() for line in MY_POSTS.splitlines() if line.strip()])
my_calls

In [ ]:
show_the_crowd(my_calls)

## Limitations

- The level labels are silver labels from a language model; the scores of chapter 8 say how well the network learned those labels, not how well it knows the levels.
- A blind re-labelling of 200 posts agrees with about 90% of the labels on the six levels but with only about a third of the `none` labels: `none` is the weak class. The posts on which two models agree are also likely the easier ones, so the scores look better than on posts in general.
- The data is English tweets only, and there is no test set from another source.
- The ladder describes what a crowd pays attention to. It does not predict prices.

***

## 局限

- 六级标签是大模型打的银标签。第 8 章的分数说的是网络把这些标签学得多像，不是它认六级认得多准。
- 盲标 200 条抽查：六级的标签约 90% 一致，`none` 这一类只有约三分之一一致，`none` 是弱项。两个模型判得一样的帖子多半也是好判的，所以分数会比一般帖子上好看。
- 数据只有英文推文，也没有别处来的测试集。
- 梯子说的是人群在关注什么，它不预测价格。